In [1]:
from langchain.schema import Document
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

from dotenv import load_dotenv
load_dotenv()
embedding_function = OpenAIEmbeddings(model="text-embedding-3-large")

from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

In [2]:
docs = [
    Document(
        page_content="Peak Performance Gym was founded in 2015 by former Olympic athlete Marcus Chen. With over 15 years of experience in professional athletics, Marcus established the gym to provide personalized fitness solutions for people of all levels. The gym spans 10,000 square feet and features state-of-the-art equipment.",
        metadata={"source": "about.txt"}
    ),
    Document(
        page_content="Peak Performance Gym is open Monday through Friday from 5:00 AM to 11:00 PM. On weekends, our hours are 7:00 AM to 9:00 PM. We remain closed on major national holidays. Members with Premium access can enter using their key cards 24/7, including holidays.",
        metadata={"source": "hours.txt"}
    ),
    Document(
        page_content="Our membership plans include: Basic (₹1,500/month) with access to gym floor and basic equipment; Standard (₹2,500/month) adds group classes and locker facilities; Premium (₹4,000/month) includes 24/7 access, personal training sessions, and spa facilities. We offer student and senior citizen discounts of 15% on all plans. Corporate partnerships are available for companies with 10+ employees joining.",
        metadata={"source": "membership.txt"}
    ),
    Document(
        page_content="Group fitness classes at Peak Performance Gym include Yoga (beginner, intermediate, advanced), HIIT, Zumba, Spin Cycling, CrossFit, and Pilates. Beginner classes are held every Monday and Wednesday at 6:00 PM. Intermediate and advanced classes are scheduled throughout the week. The full schedule is available on our mobile app or at the reception desk.",
        metadata={"source": "classes.txt"}
    ),
    Document(
        page_content="Personal trainers at Peak Performance Gym are all certified professionals with minimum 5 years of experience. Each new member receives a complimentary fitness assessment and one free session with a trainer. Our head trainer, Neha Kapoor, specializes in rehabilitation fitness and sports-specific training. Personal training sessions can be booked individually (₹800/session) or in packages of 10 (₹7,000) or 20 (₹13,000).",
        metadata={"source": "trainers.txt"}
    ),
    Document(
        page_content="Peak Performance Gym's facilities include a cardio zone with 30+ machines, strength training area, functional fitness space, dedicated yoga studio, spin class room, swimming pool (25m), sauna and steam rooms, juice bar, and locker rooms with shower facilities. Our equipment is replaced or upgraded every 3 years to ensure members have access to the latest fitness technology.",
        metadata={"source": "facilities.txt"}
    )
]

In [3]:
new_db = FAISS.load_local("faiss_index", embedding_function,allow_dangerous_deserialization=True)
retriever = new_db.as_retriever(search_type="mmr", search_kwargs = {"k": 3})


In [4]:
retriever.invoke("Who is the owner and what are the timings?")

[Document(id='fa868e19-5d51-4710-9eaf-b0d7ad0e57fb', metadata={'source': 'hours.txt'}, page_content='Peak Performance Gym is open Monday through Friday from 5:00 AM to 11:00 PM. On weekends, our hours are 7:00 AM to 9:00 PM. We remain closed on major national holidays. Members with Premium access can enter using their key cards 24/7, including holidays.'),
 Document(id='ee3297c5-bad8-4431-8304-f0b9e37d9c68', metadata={'source': 'membership.txt'}, page_content='Our membership plans include: Basic (₹1,500/month) with access to gym floor and basic equipment; Standard (₹2,500/month) adds group classes and locker facilities; Premium (₹4,000/month) includes 24/7 access, personal training sessions, and spa facilities. We offer student and senior citizen discounts of 15% on all plans. Corporate partnerships are available for companies with 10+ employees joining.'),
 Document(id='ec6455d7-fcf6-4170-9772-199c3b32e740', metadata={'source': 'classes.txt'}, page_content='Group fitness classes at Pe

In [ ]:
template = """ 
Answer the question based only on the following context: {context}
Question: {question}
"""
prompt=ChatPromptTemplate.from_template(template)
llm = ChatOpenAI(model="gpt-4o")



rag_chain=prompt|llm

In [6]:
from langchain_core.messages import AIMessage,HumanMessage,BaseMessage
from typing import TypedDict,Annotated,List
from langgraph.graph import StateGraph,END,START
from pydantic import BaseModel,Field
from langchain.schema import Document

In [7]:
class Agentstate(TypedDict):
    messages:List[BaseMessage]
    documents:List[Document]
    Topic:str

In [8]:
class GradeQuestion(BaseModel):
    score:str=Field(description="Question is about gym? If yes -> 'Yes' if not -> 'No' ")

def qn_classifier(state:Agentstate):
    qn= state["messages"][-1].content
    system = """ You are a classifier that determines whether a user's question is about one of the following topics 
    
    1. Gym History & Founder
    2. Operating Hours
    3. Membership Plans 
    4. Fitness Classes
    5. Personal Trainers
    6. Facilities & Equipment
    
    If the question IS about any of these topics, respond with 'Yes'. Otherwise, respond with 'No'.

    """
    qn_prompt=ChatPromptTemplate.from_messages([
        ("system",system),
        ("human","user question:{question}")

    ])

    llm = ChatOpenAI(model="gpt-4o")
    structured_llm=llm.with_structured_output(GradeQuestion)
    chain=qn_prompt|structured_llm
    result=chain.invoke({"question":qn})
    state["Topic"]=result.score
    return state

In [ ]:
def Topic_router(state:Agentstate):
    on_topic=state["Topic"]
   
    if on_topic.lower()=="yes":

        return "on_topic"
    
    return "off_topic"

def retrieve(state:Agentstate):
    print("starttt")
    qn=state["messages"][-1].content
    documents=retriever.invoke(qn)
    state["documents"]=documents
    return state

def generate_answer(state:Agentstate):
    qn=state["messages"][-1].content
    documents=state["documents"]
    result=rag_chain.invoke({"context":documents,"question":qn})
    state["messages"].append(result)
    return state

def off_topic(state:Agentstate):
    messages=AIMessage(content="sorry, I cant answer")
    state["messages"].append(messages)
    return state
    


In [21]:
workflow=StateGraph(Agentstate)

workflow.add_node("qn_classifier",qn_classifier)
workflow.add_node("Topic_router",Topic_router)
workflow.add_node("off_topic_respose",off_topic)
workflow.add_node("retrieve",retrieve)
workflow.add_node("generate_answer",generate_answer)
workflow.set_entry_point("qn_classifier")
# workflow.add_edge("qn_classifier","Topic_router")
workflow.add_conditional_edges("qn_classifier",Topic_router,
                               {"on_topic":"retrieve",
                                "off_topic":"off_topic_respose"

                               })
workflow.add_edge("retrieve","generate_answer")
workflow.add_edge("generate_answer",END)
workflow.add_edge("off_topic_respose",END)

app=workflow.compile()



In [22]:
app.invoke(input={
    "messages":[HumanMessage(content="Who is the owner and what are the timings?")]
})

llll
starttt


{'messages': [HumanMessage(content='Who is the owner and what are the timings?', additional_kwargs={}, response_metadata={}),
  AIMessage(content="The provided context does not include information about the owner of Peak Performance Gym. The gym's timings are as follows: open Monday through Friday from 5:00 AM to 11:00 PM, on weekends from 7:00 AM to 9:00 PM, and closed on major national holidays. However, members with Premium access can enter 24/7, including holidays.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 78, 'prompt_tokens': 374, 'total_tokens': 452, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_1827dd0c55', 'id': 'chatcmpl-CEHfAE1ohWMQmEMTLMVcm9Fex9UOA', 'service_tier': 'default', 'finish_reason': 'stop', 'log